In [ ]:
# Run this cell to install ChromaDB if desired
try:
    assert version('chromadb') == '0.4.17'
except:
    !pip install chromadb==0.4.17
try:
    assert version('pysqlite3') == '0.5.2'
except:
    !pip install pysqlite3-binary==0.5.2
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

import chromadb

##Load the dataset
Load data and perform basic data checks to ensure you are using relevant data for the analysis

In [ ]:
# Load the dataset
import pandas as pd

reviews = pd.read_csv("womens_clothing_e-commerce_reviews.csv")

# Display the first few entries
reviews.head()

In [ ]:
# Display all the columns in the dataset
reviews.columns

##Import all the modules

In [ ]:
import os
import openai
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from scipy.spatial import distance
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

##Initialize the Client of the OpenAI, create an embeddings and store them into a list called "embeddings"

In [ ]:
import openai

# Initialize a client
client = openai.OpenAI()

# Create a list of reviews extract from the dataset reviews
reviews_descriptions = reviews['Review Text'].dropna()

# Create an embedding for each review in reviews
response = client.embeddings.create(
    model = "text-embedding-3-small", # <--- In this case I chose to use this model by OpenAI
    input = reviews_descriptions
)
response_dict = response.model_dump()

# Extract embeddings from the response
embeddings = [item["embedding"] for item in response_dict["data"]]
#print(embeddings[0])

##Apply the t-SNE (t-Distributed Stochastic Neighbor)

In [ ]:
#Apply the t-SNE for dimensionality reduction. The goal is to reduct to a 2-dimensional numpy array the vectors dimensions

def dim_reduct(embeddings):
    tsne = TSNE(n_components = 2, random_state = 101)
    return tsne.fit_transform(embeddings)

embeddings_2d = dim_reduct(np.array(embeddings))

##Plot the visualization of reviews embeddings with Matplotlib

In [ ]:
def plot_scat(tsne_results):
    plt.figure(figsize = (12,8))
    for i, point in enumerate(tsne_results):
        plt.scatter(point[0], point[1], alpha=0.5)
        plt.text(point[0], point[1], str(i), fontsize=8, verticalalignment='center')
    plt.title("t-SNE Visualization of Review Embeddings")
    plt.xlabel("t-SNE feature 1")
    plt.ylabel("t-SNE feature 2")
    plt.show()

plot_scat(embeddings_2d)

##Feedback categorization

Use the embeddings to identify some reviews that discuss topics such as 'quality', 'fit', 'style', 'comfort'.

In [ ]:
# Define the topics of some reviews
topics = ['quality', 'fit', 'style', 'comfort']

# Generate the embeddings of the topics
topics_response = client.embeddings.create(
    model = "text-embedding-3-small",
    input = topics
)
topics_response_dict = topics_response.model_dump()

# Save the topics embeddings into a list
topics_embeddings = [topic["embedding"] for topic in topics_response_dict["data"]]

##Create a find_closest() function to check the similarity between embeddings

In [ ]:
def find_closest(topics_embeddings, embedding):
    from scipy.spatial import distance
    distances = []
    for index, topic_embedding in enumerate(topics_embeddings):
        dist = distance.cosine(topic_embedding, embedding)
        distances.append({"Distanza": dist, "Indice": index})
    return min(distances, key=lambda x: x["Distanza"])

feedback_categorize = [find_closest(topics_embeddings, embedding) for embedding in embeddings]
print(feedback_categorize[0:10])

##Initialize ChromaDB vector database

In [ ]:
# Vector database, create a persisten client

client = chromadb.PersistentClient()

##Create a new collection

In [ ]:
# Create a new collection

reviews_clothes_db = client.create_collection(
    name = "reviews_database",
    embedding_function = OpenAIEmbeddingFunction(model_name = "text-embedding-3-small",
                                                 api_key = os.environ["OPENAI_API_KEY"])
)

##Add reviews into ChromaDB

In [ ]:
# Add reviews into the database

# Convert reviews_descriptions to a list if it's a pandas Series
if hasattr(reviews_descriptions, "tolist"):
    reviews_descriptions_list = reviews_descriptions.tolist()
else:
    reviews_descriptions_list = reviews_descriptions

reviews_clothes_db.add(
    documents=reviews_descriptions_list,
    ids=[str(i) for i in range(len(reviews_descriptions_list))]
)

##Retrieve the reviews and query the database

In [ ]:
# Retrieve the reviews_db and query the database
def compare_funct(input_text, vector_db, n):
    collection = client.get_collection(
    name = "reviews_database",
    embedding_function = OpenAIEmbeddingFunction(model_name = "text-embedding-3-small",
                                                 api_key = os.environ["OPENAI_API_KEY"]))
    result = collection.query(
        query_texts = [input_text],
        n_results = n
        )
    return result

##Try to query the database!

In [ ]:
example = "Absolutely wonderful - silky and sexy and comfortable" # Or anything else you want to search

# Pass the input as a string, not as a list of strings
most_similar_sentence = compare_funct(example, reviews_clothes_db, 2)["documents"][0]

print(f"These are the most semantically similar sentences: {example}: \n {most_similar_sentence}")